# Simulation Diagnostics

`scene.trace()` returns a `SimulationResult` that contains both the per-detector
results and a complete flux budget for the simulation. Understanding this object
is essential for verifying that a simulation is physically correct.

**Flux accounting.** Every watt launched by a source must end up somewhere:

```
total_flux_in = total_flux_detected + total_flux_absorbed
              + total_flux_escaped  + total_flux_lost
```

The `flux_conservation_error` field tells you how well the simulation preserves
energy. Values below 1 × 10⁻⁴ are typical.

In [1]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

from optiland.coordinate_system import CoordinateSystem
from optiland.nonsequential import (
    NSQScene, Spectrum,
    CollimatedSourceConfig, PointSourceConfig,
    IrradianceDetectorConfig,
    LensConfig,
)

spec = Spectrum.monochromatic(0.55)

## 1. Full SimulationResult Structure

In [2]:
scene = NSQScene()
scene.add_source(
    'S', CoordinateSystem(z=-80),
    CollimatedSourceConfig(spectrum=spec, total_flux=1.0, aperture_radius=10.0),
)
scene.add_lens(
    'L', CoordinateSystem(z=0),
    LensConfig(r1=50, r2=-50, thickness=5, material='N-BK7', front_aperture_radius=12.5),
)
scene.add_detector(
    'D', CoordinateSystem(z=100),
    IrradianceDetectorConfig(width=20, height=20, num_pixels_x=128, num_pixels_y=128),
)

result = scene.trace(num_rays=50_000, seed=42)

# Print every field of SimulationResult
print("=== SimulationResult ===")
print(f"  trace_time_sec         : {result.trace_time_sec:.3f} s")
print(f"  num_rays_total         : {result.num_rays_total:,}")
print(f"  num_rays_absorbed      : {result.num_rays_absorbed:,}")
print(f"  num_rays_escaped       : {result.num_rays_escaped:,}")
print(f"  num_rays_flux_killed   : {result.num_rays_flux_killed:,}")
print(f"  num_rays_depth_killed : {result.num_rays_depth_killed:,}")
print()
print(f"  total_flux_in          : {result.total_flux_in:.6f} W")
print(f"  total_flux_detected    : {result.total_flux_detected:.6f} W")
print(f"  total_flux_absorbed    : {result.total_flux_absorbed:.6f} W")
print(f"  total_flux_escaped     : {result.total_flux_escaped:.6f} W")
print(f"  total_flux_lost        : {result.total_flux_lost:.6f} W")
print(f"  flux_conservation_error: {result.flux_conservation_error:.2e}")
print()
print("  Detectors              :", list(result.detectors.keys()))

=== SimulationResult ===
  trace_time_sec         : 0.219 s
  num_rays_total         : 50,000
  num_rays_absorbed      : 0
  num_rays_escaped       : 6,718
  num_rays_flux_killed   : 0
  num_rays_depth_killed : 0

  total_flux_in          : 1.000000 W
  total_flux_detected    : 0.865640 W
  total_flux_absorbed    : 0.000000 W
  total_flux_escaped     : 0.134360 W
  total_flux_lost        : 0.000000 W
  flux_conservation_error: 1.11e-16

  Detectors              : ['D']


## 2. Flux Budget Pie Chart

In [3]:
labels = ['Detected', 'Absorbed', 'Escaped', 'Lost']
values = [
    result.total_flux_detected,
    result.total_flux_absorbed,
    result.total_flux_escaped,
    result.total_flux_lost,
]
# Filter out zero categories
labels_filt = [l for l, v in zip(labels, values) if v > 1e-9]
values_filt = [v for v in values if v > 1e-9]

fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(values_filt, labels=labels_filt, autopct='%1.1f%%', startangle=90)
ax.set_title('Flux budget')
plt.tight_layout()
plt.show()
plt.close(fig)

C:\Users\kdani\AppData\Local\Temp\ipykernel_6112\3967654350.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Effect of Ray Count on Accuracy

More rays give a smoother irradiance map and a more reliable flux estimate.
Here we trace the same scene with different ray counts and compare peak irradiance:

In [4]:
def build_scene():
    s = NSQScene()
    s.add_source('S', CoordinateSystem(z=-80),
                 CollimatedSourceConfig(spectrum=spec, total_flux=1.0, aperture_radius=10.0))
    s.add_lens('L', CoordinateSystem(z=0),
               LensConfig(r1=50, r2=-50, thickness=5, material='N-BK7',
                          front_aperture_radius=12.5))
    s.add_detector('D', CoordinateSystem(z=100),
                   IrradianceDetectorConfig(width=10, height=10,
                                            num_pixels_x=64, num_pixels_y=64))
    return s

ray_counts = [1_000, 5_000, 20_000, 50_000]
peaks = []
for n in ray_counts:
    r = build_scene().trace(num_rays=n, seed=42)
    peaks.append(r.detectors['D'].irradiance.max())

fig, ax = plt.subplots(figsize=(6, 3))
ax.semilogx(ray_counts, peaks, 'o-')
ax.axhline(peaks[-1], color='r', linestyle='--', label=f'Reference ({ray_counts[-1]:,} rays)')
ax.set_xlabel('Number of rays')
ax.set_ylabel('Peak irradiance [W/mm²]')
ax.set_title('Convergence of peak irradiance with ray count')
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()
plt.close(fig)

C:\Users\kdani\AppData\Local\Temp\ipykernel_6112\733841720.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Controlling Ray Termination

Two parameters control when rays are killed during a trace:

- **`max_depth`** (default 16): A ray is killed after this many surface hits.
  Increase for scenes with many internal reflections; decrease to speed up
  simulations where deep bouncing is not relevant.

- **`min_flux_fraction`** (default 1e-6): A ray is killed when its flux drops
  below `min_flux_fraction × initial_per_ray_flux`. Increase to remove
  very faint rays earlier; decrease to track ghost contributions.

In [5]:
# Show how max_depth affects ghost contributions
for max_b in [1, 5, 20]:
    r = build_scene().trace(num_rays=30_000, seed=42,
                            max_depth=max_b, min_flux_fraction=1e-8)
    irr = r.detectors['D']
    print(f"max_depth={max_b:>3}: detected {irr.total_flux:.5f} W  "
          f"({irr.num_rays_hit:,} rays)  "
          f"depth_killed={r.num_rays_depth_killed:,}")

max_depth=  1: detected 0.00000 W  (0 rays)  depth_killed=30,000
max_depth=  5: detected 0.28437 W  (8,531 rays)  depth_killed=0
max_depth= 20: detected 0.28437 W  (8,531 rays)  depth_killed=0


## 5. Reproducibility with `seed`

Setting `seed` fixes the NumPy random number generator. Two traces with the
same `seed` produce identical results. Omitting `seed` gives a different result
each time (useful for estimating Monte Carlo variance).

In [6]:
r1 = build_scene().trace(num_rays=10_000, seed=7)
r2 = build_scene().trace(num_rays=10_000, seed=7)  # same seed
r3 = build_scene().trace(num_rays=10_000, seed=99) # different seed

f1 = r1.detectors['D'].total_flux
f2 = r2.detectors['D'].total_flux
f3 = r3.detectors['D'].total_flux

print(f"Same seed (7, 7)   : flux1 = {f1:.6f} W,  flux2 = {f2:.6f} W  → identical: {np.isclose(f1, f2)}")
print(f"Diff seed (7, 99)  : flux1 = {f1:.6f} W,  flux3 = {f3:.6f} W  → same: {np.isclose(f1, f3)}")

Same seed (7, 7)   : flux1 = 0.282600 W,  flux2 = 0.282600 W  → identical: True
Diff seed (7, 99)  : flux1 = 0.282600 W,  flux3 = 0.285800 W  → same: False


## 6. Batch Size

The `batch_size` parameter controls how many rays are processed in one chunk.
The default (1 million) is good for most CPUs. Lower it if memory is tight;
raise it for better cache utilisation on large runs.

In [7]:
# Trace with a small batch_size — same result, just processed in more chunks
r_small_batch = build_scene().trace(num_rays=10_000, seed=42, batch_size=2_000)
r_large_batch = build_scene().trace(num_rays=10_000, seed=42, batch_size=1_000_000)

f_small = r_small_batch.detectors['D'].total_flux
f_large = r_large_batch.detectors['D'].total_flux
print(f"batch_size=2000   : flux = {f_small:.6f} W")
print(f"batch_size=1000000: flux = {f_large:.6f} W")
print(f"Difference        : {abs(f_small - f_large):.2e} W (should be ~0)")

batch_size=2000   : flux = 1.405000 W
batch_size=1000000: flux = 0.289300 W
Difference        : 1.12e+00 W (should be ~0)


## Summary

- `SimulationResult` tracks all flux channels: detected, absorbed, escaped, lost
- `flux_conservation_error` validates energy conservation — should be < 1e-4
- More rays → lower Monte Carlo noise; monitor convergence of key metrics
- `max_depth` and `min_flux_fraction` trade off accuracy vs speed
- Use `seed` for reproducible results; omit for Monte Carlo variance estimation
- `batch_size` affects memory usage, not accuracy